# 0. Setup

In [ ]:
import ibis
from ibis import _
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str, reindex_entity, add_fe

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

# First difference the panel data

In [ ]:
g_name = 'working_yearly_g'
n_name = 'working_yearly_n'
t_panel_g = con.table(g_name)
t_panel_n = con.table(n_name)

#--- #
t_panel = add_fe(t_panel_g, fe=['t', 'i', 'c'])
t_panel_sample = (
    t_panel
    .select([c for c in t_panel.columns if c.startswith('year_')])
    .order_by(ibis.random())
    .limit(5)
)
display(t_panel_sample.execute())

2006 year_2006 ('registered_number', 'year', 'gva', 'total_assets', 'employees', 'y', 'k', 'l', 'wg1_y', 'wg2_y', 'wg3_y', 'wg1_k', 'wg2_k', 'wg3_k', 'wg1_l', 'wg2_l', 'wg3_l', 'w2g1_k', 'wg2wg1_k', 'wg3wg1_k', 'w2g1_l', 'wg2wg1_l', 'wg3wg1_l', 'wg1wg2_k', 'w2g2_k', 'wg3wg2_k', 'wg1wg2_l', 'w2g2_l', 'wg3wg2_l', 'wg1wg3_k', 'wg2wg3_k', 'w2g3_k', 'wg1wg3_l', 'wg2wg3_l', 'w2g3_l', 'w3g1_k', 'w3g1_l', '(i-wg1)w2g1_k', '(i-vg1)w2g1_k', '(i-wg1)w2g1_l', '(i-vg1)w2g1_l', 'w4g1_k', 'w4g1_l', '(i-wg1)w3g1_k', '(i-vg1)w3g1_k', '(i-wg1)w3g1_l', '(i-vg1)w3g1_l', 'year__2019', 'year__2018', 'year__2021', 'year__2023', 'year__2024', 'year__2008', 'year__2016', 'year__2020', 'year__2010', 'year__2009', 'year__2015', 'year__2014', 'year__2012', 'year__2011', 'year__2013', 'year__2007', 'year__2022', 'year__2006', 'year__2017')


,year__2019,year__2018,year__2021,year__2023,year__2024,year__2008,year__2016,year__2020,year__2010,year__2009,year__2015,year__2014,year__2012,year__2011,year__2013,year__2007,year__2022,year__2006,year__2017
0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0


# 1a. LMM
- 26 seconds run nowadays estimating 2 models

In [4]:
panel_name = "working_yearly_g"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models_1 = {
    'lmm_exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i', 'c'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l'
            ],
        },
        'description': 'Instrument endogenous effect, LMM'
    }
}
models_3 = {
    # 'lmm_3_rings': {
    #     'Y': 'y',
    #     'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
    #     'W': ['wg2_y', 'wg2_k', 'wg2_l', 'wg3_l', 'wg3_k', 'wg3_y'],
    #     'fe': ['t', 'i', 'c'],
    #     'description': 'Strict exogeneity, structural form'
    # },
    'lmm_3_rings_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 'c', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l']
        },
        'description': '3-level regressors, squared instruments'
    },
    'lmm_3_rings_more_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['t', 'i', 'c'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l', 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l', 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l', 'wg3wg2_k', 'wg3wg2_l']
        },
        'description': '3-level regressors, many instruments'
    }
}

models = models_3
# out_name = "results_1a_lmm_pc8"
out_name = "results_1b_lmm_3r"

# 5m run for 3 complicated IV models
run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

Running model 'lmm_3_rings_instr' as linearmodels panel IV, formula: y ~ k + l + wg1_k + wg1_l + wg2_k + wg2_l + wg3_k + wg3_l + year__2017 + year__2006 + year__2011 + year__2016 + year__2023 + year__2021 + year__2024 + year__2020 + year__2015 + year__2014 + year__2018 + year__2019 + year__2013 + year__2022 + year__2007 + year__2008 + year__2010 + year__2009 + year__2012 + [wg1_y + wg2_y + wg3_y ~ w2g1_k + w2g1_l + w2g2_k + w2g2_l + w2g3_k + w2g3_l]
Index(['registered_number', 'year', 'y', 'k', 'l', 'wg1_y', 'wg1_k', 'wg1_l',
       'wg2_y', 'wg2_k', 'wg2_l', 'wg3_y', 'wg3_k', 'wg3_l', 'w2g1_k',
       'w2g1_l', 'w2g2_k', 'w2g2_l', 'w2g3_k', 'w2g3_l', 'year__2017',
       'year__2006', 'year__2011', 'year__2016', 'year__2023', 'year__2021',
       'year__2024', 'year__2020', 'year__2015', 'year__2014', 'year__2018',
       'year__2019', 'year__2013', 'year__2022', 'year__2007', 'year__2008',
       'year__2010', 'year__2009', 'year__2012', 'const'],
      dtype='str')
❌ Model 'lmm_3_ri

Traceback (most recent call last):
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\formulaic\materializers\base.py", line 559, in _evaluate_factor
    value, variables = self._lookup(factor.expr)
                       ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\formulaic\materializers\base.py", line 628, in _lookup
    raise NameError(
        f"`{name}` is not present in the dataset or evaluation context."
    )
NameError: `y` is not present in the dataset or evaluation context.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\lazyst\Files\ucl\Dissertation\model\src\f_7_run_panel.py", line 194, in run_panel
    mod_iv = IVGMMCUE.from_formula(
        formula=formula_str,
        data=(df_model)
    )
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\linearmodels\iv\model.py", line 1453, in from_formul

Running model 'lmm_3_rings_more_instr' as linearmodels panel IV, formula: y ~ k + l + wg1_k + wg1_l + wg2_k + wg2_l + wg3_k + wg3_l + year__2018 + year__2019 + year__2014 + year__2015 + year__2021 + year__2023 + year__2024 + year__2013 + year__2008 + year__2016 + year__2020 + year__2011 + year__2007 + year__2022 + year__2012 + year__2006 + year__2017 + year__2009 + year__2010 + [wg1_y + wg2_y + wg3_y ~ w2g1_k + w2g1_l + wg1wg2_k + wg1wg2_l + wg1wg3_k + wg1wg3_l + w2g2_k + w2g2_l + wg2wg3_k + wg2wg3_l + wg2wg1_k + wg2wg1_l + w2g3_k + w2g3_l + wg3wg1_k + wg3wg1_l + wg3wg2_k + wg3wg2_l]
Index(['registered_number', 'year', 'y', 'k', 'l', 'wg1_y', 'wg1_k', 'wg1_l',
       'wg2_y', 'wg2_k', 'wg2_l', 'wg3_y', 'wg3_k', 'wg3_l', 'w2g1_k',
       'w2g1_l', 'wg1wg2_k', 'wg1wg2_l', 'wg1wg3_k', 'wg1wg3_l', 'w2g2_k',
       'w2g2_l', 'wg2wg3_k', 'wg2wg3_l', 'wg2wg1_k', 'wg2wg1_l', 'w2g3_k',
       'w2g3_l', 'wg3wg1_k', 'wg3wg1_l', 'wg3wg2_k', 'wg3wg2_l', 'year__2018',
       'year__2019', 'year__201

Traceback (most recent call last):
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\formulaic\materializers\base.py", line 559, in _evaluate_factor
    value, variables = self._lookup(factor.expr)
                       ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\formulaic\materializers\base.py", line 628, in _lookup
    raise NameError(
        f"`{name}` is not present in the dataset or evaluation context."
    )
NameError: `y` is not present in the dataset or evaluation context.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\lazyst\Files\ucl\Dissertation\model\src\f_7_run_panel.py", line 194, in run_panel
    mod_iv = IVGMMCUE.from_formula(
        formula=formula_str,
        data=(df_model)
    )
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\linearmodels\iv\model.py", line 1453, in from_formul

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [ ]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

panel_name = "working_yearly_n"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
m_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't', 'c'],
        'description': 'Distance 1 model'
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't', 'c'],
        'description': 'Distance 2 model'
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't', 'c'],
        'description': 'Distance 3 model'
    }
}
deeper_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't', 'c'],
        'description': '2nd-order spatial lag model'
    },
    'dd1-3lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't', 'c'],
        'description': '3rd-order spatial lag model'
    }
}

models = m_models
out_name = "results_2a_dd"

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel((mod, df_diff, model_name))
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lazyst\\Files\\ucl\\Dissertation\\model\\tmp\\working_yearly_n_diff.parquet'